In [ ]:
!pip -q install wfdb

import os, glob, ast, copy, json, math, random, time, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import wfdb

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
FIG = "/content/figs"; os.makedirs(FIG, exist_ok=True)
def savefig(name): plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight")

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
SEED = 42
seed_everything(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = device == "cuda"
print("device:", device, "|", torch.cuda.get_device_name(0) if USE_AMP else "CPU (enable a GPU runtime!)")

In [ ]:
QUICK = False   # True = 2 epochs everywhere, only to smoke-test the pipeline

CFG = dict(epochs_sup=20, epochs_ssl=40, epochs_ft=15,
           batch=128, batch_ssl=256,
           lr_sup=1e-3, lr_ft=5e-4, lr_ssl=1e-3, workers=2)
if QUICK:
    CFG.update(epochs_sup=2, epochs_ssl=2, epochs_ft=2)

CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
K = len(CLASSES)
results = {}     # everything we want to quote on the resume gets stored here

In [ ]:
ZIP_URL = ("https://physionet.org/static/published-projects/ptb-xl/"
           "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip")
if not glob.glob("/content/ptbxl_tmp/*/ptbxl_database.csv"):
    !wget -q --show-progress -O /content/ptbxl.zip {ZIP_URL}
    !unzip -q /content/ptbxl.zip "*records100/*" "*ptbxl_database.csv" "*scp_statements.csv" -d /content/ptbxl_tmp
    !rm -f /content/ptbxl.zip
ROOT = glob.glob("/content/ptbxl_tmp/*/")[0]
print("data root:", ROOT)

In [ ]:
db = pd.read_csv(ROOT + "ptbxl_database.csv", index_col="ecg_id")
db["scp_codes"] = db["scp_codes"].apply(ast.literal_eval)

agg = pd.read_csv(ROOT + "scp_statements.csv", index_col=0)
agg = agg[agg["diagnostic"] == 1]                       # diagnostic statements only

def to_superclasses(codes):
    return sorted({agg.loc[c, "diagnostic_class"] for c in codes if c in agg.index})

db["superclasses"] = db["scp_codes"].apply(to_superclasses)
Y_all = np.array([[c in s for c in CLASSES] for s in db["superclasses"]], dtype=np.float32)
keep = Y_all.sum(1) > 0                                  # drop records with no diagnostic label
meta = db[keep].reset_index()                            # keeps ecg_id, patient_id, age, sex, strat_fold, filename_lr ...
Y = Y_all[keep]
print(f"records kept: {len(meta):,} of {len(db):,}")
print("prevalence:", dict(zip(CLASSES, Y.mean(0).round(3))))

In [ ]:
from concurrent.futures import ThreadPoolExecutor
CACHE = "/content/ptbxl_X100.npy"

def load_one(f):
    return wfdb.rdsamp(os.path.join(ROOT, f))[0].astype(np.float32)   # (1000, 12)

if os.path.exists(CACHE):
    X = np.load(CACHE)
else:
    with ThreadPoolExecutor(8) as ex:
        X = np.stack(list(tqdm(ex.map(load_one, meta["filename_lr"]), total=len(meta), desc="reading ECGs")))
    np.save(CACHE, X)
print("signals:", X.shape)      # (N, 1000, 12)

In [ ]:
folds = meta["strat_fold"].values
tr, va, te = folds <= 8, folds == 9, folds == 10

X = np.ascontiguousarray(X.transpose(0, 2, 1))          # -> (N, 12, 1000)
mu = X[tr].mean((0, 2), keepdims=True); sd = X[tr].std((0, 2), keepdims=True) + 1e-6
X -= mu; X /= sd                                         # per-lead z-score, statistics from TRAIN only

Xtr, Ytr = X[tr], Y[tr]
Xva, Yva = X[va], Y[va]
Xte, Yte = X[te], Y[te]
meta_te = meta[te].reset_index(drop=True)
print(f"train {len(Xtr):,} | val {len(Xva):,} | test {len(Xte):,}")

In [ ]:
def augment(x, strength=1.0):
    C, T = x.shape
    x = x * (1 + 0.2 * strength * (2 * torch.rand(C, 1) - 1))                      # per-lead gain
    x = x + 0.1 * strength * torch.rand(1) * torch.randn(C, T)                     # sensor noise
    t = torch.arange(T) / 100.0                                                     # 100 Hz
    f, ph = 0.1 + 0.4 * torch.rand(1), 2 * math.pi * torch.rand(1)
    x = x + 0.3 * strength * torch.rand(1) * torch.sin(2 * math.pi * f * t + ph)   # baseline wander
    x = torch.roll(x, int(torch.randint(-int(100 * strength) - 1, int(100 * strength) + 2, (1,))), 1)
    if torch.rand(1) < 0.3 * strength:                                              # lead dropout
        x[torch.randperm(C)[: int(torch.randint(1, 4, (1,)))]] = 0
    if torch.rand(1) < 0.5 * strength:                                              # time masking
        L = int(torch.randint(50, 151, (1,))); s = int(torch.randint(0, T - L, (1,))); x[:, s:s + L] = 0
    return x

class ECGSet(Dataset):
    def __init__(self, X, Y, strength=0.0): self.X, self.Y, self.s = X, Y, strength
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i])
        if self.s > 0: x = augment(x, self.s)
        return x, torch.from_numpy(self.Y[i])

class TwoViews(Dataset):                      # for contrastive pre-training (labels never used)
    def __init__(self, X): self.X = X
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i]); return augment(x, 1.0), augment(x, 1.0)

def loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=min(bs, len(ds)), shuffle=shuffle, drop_last=shuffle,
                      num_workers=CFG["workers"], pin_memory=USE_AMP, persistent_workers=CFG["workers"] > 0)

In [ ]:
class Block(nn.Module):
    def __init__(s, cin, cout, stride):
        super().__init__()
        s.c1 = nn.Conv1d(cin, cout, 7, stride, 3, bias=False); s.b1 = nn.BatchNorm1d(cout)
        s.c2 = nn.Conv1d(cout, cout, 7, 1, 3, bias=False);     s.b2 = nn.BatchNorm1d(cout)
        s.sc = None if (cin == cout and stride == 1) else nn.Sequential(
            nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))
    def forward(s, x):
        o = s.b2(s.c2(F.relu(s.b1(s.c1(x)))))
        return F.relu(o + (x if s.sc is None else s.sc(x)))

class ResNet1D(nn.Module):
    def __init__(s, n_out=K, in_ch=12, widths=(64, 128, 256, 256)):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(in_ch, widths[0], 15, 2, 7, bias=False), nn.BatchNorm1d(widths[0]), nn.ReLU())
        layers, c = [], widths[0]
        for w in widths:
            layers += [Block(c, w, 2), Block(w, w, 1)]; c = w
        s.body = nn.Sequential(*layers); s.head = nn.Linear(c, n_out)
    def forward(s, x): return s.head(s.body(s.stem(x)).mean(-1))

class CNNTransformer(nn.Module):
    def __init__(s, n_out=K, in_ch=12, d=128, heads=4, layers=4, drop=0.1, max_tokens=125):
        super().__init__()
        s.d = d
        s.stem = nn.Sequential(
            nn.Conv1d(in_ch, 64, 15, 2, 7, bias=False), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d, 7, 2, 3, bias=False),      nn.BatchNorm1d(d),  nn.GELU(),
            nn.Conv1d(d, d, 7, 2, 3, bias=False),       nn.BatchNorm1d(d),  nn.GELU())   # 1000 -> 125 tokens
        s.pos = nn.Parameter(torch.zeros(1, max_tokens, d)); nn.init.trunc_normal_(s.pos, std=0.02)
        s.cls = nn.Parameter(torch.zeros(1, 1, d));          nn.init.trunc_normal_(s.cls, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, activation="gelu", batch_first=True, norm_first=True)
        s.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        s.norm = nn.LayerNorm(d); s.head = nn.Linear(d, n_out)
    def encode(s, x):
        t = s.stem(x).transpose(1, 2)                         # (B, L, d)
        t = t + s.pos[:, : t.shape[1]]
        t = torch.cat([s.cls.expand(len(t), -1, -1), t], 1)
        return s.norm(s.enc(t)).mean(1)                       # (B, d) pooled representation
    def forward(s, x): return s.head(s.encode(x))

print("ResNet1D params:      %.2fM" % (sum(p.numel() for p in ResNet1D().parameters()) / 1e6))
print("CNNTransformer params: %.2fM" % (sum(p.numel() for p in CNNTransformer().parameters()) / 1e6))

In [ ]:
@torch.no_grad()
def predict_logits(model, X, bs=512):
    model.eval().to(device); out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(np.ascontiguousarray(X[i:i + bs])).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out.append(model(xb).float().cpu())
    return torch.cat(out).numpy()

sigmoid = lambda z: 1 / (1 + np.exp(-z))

def macro_auc(y, logits, per_class=False):
    aucs = [roc_auc_score(y[:, c], logits[:, c]) if len(np.unique(y[:, c])) > 1 else np.nan for c in range(K)]
    return aucs if per_class else float(np.nanmean(aucs))

def train_supervised(model, Xa, Ya, Xv, Yv, epochs, lr, tag, aug=0.5, wd=1e-2):
    model.to(device)
    dl = loader(ECGSet(Xa, Ya, aug), CFG["batch"], True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(dl), pct_start=0.2)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best, best_state, hist = -1, None, []
    for ep in range(epochs):
        model.train(); tot = 0
        for xb, yb in dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = F.binary_cross_entropy_with_logits(model(xb).float(), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
        auc = macro_auc(Yv, predict_logits(model, Xv)); hist.append((tot / len(dl), auc))
        if auc > best: best, best_state = auc, copy.deepcopy(model.state_dict())
        if (ep + 1) % max(1, epochs // 5) == 0 or ep == epochs - 1:
            print(f"[{tag}] ep {ep+1:>3}/{epochs}  loss {tot/len(dl):.4f}  val macro-AUROC {auc:.4f}")
    model.load_state_dict(best_state)
    return hist

In [ ]:
seed_everything(SEED)
resnet = ResNet1D()
train_supervised(resnet, Xtr, Ytr, Xva, Yva, CFG["epochs_sup"], CFG["lr_sup"], "ResNet1D", aug=0.5)
lg_resnet = predict_logits(resnet, Xte)
results["auc_resnet_test"] = macro_auc(Yte, lg_resnet)
print(f"\nResNet1D  test macro-AUROC: {results['auc_resnet_test']:.4f}")

In [ ]:
def nt_xent(z1, z2, tau=0.2):
    n = len(z1); z = F.normalize(torch.cat([z1, z2]).float(), dim=1)
    sim = (z @ z.T) / tau
    sim = sim.masked_fill(torch.eye(2 * n, dtype=torch.bool, device=z.device), -1e9)
    tgt = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device)
    return F.cross_entropy(sim, tgt)

seed_everything(SEED)
enc = CNNTransformer().to(device)
proj = nn.Sequential(nn.Linear(enc.d, enc.d), nn.ReLU(), nn.Linear(enc.d, 64)).to(device)
dl = loader(TwoViews(Xtr), CFG["batch_ssl"], True)
params = list(enc.parameters()) + list(proj.parameters())
opt = torch.optim.AdamW(params, lr=CFG["lr_ssl"], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG["lr_ssl"], total_steps=CFG["epochs_ssl"] * len(dl), pct_start=0.1)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

ssl_hist = []
for ep in range(CFG["epochs_ssl"]):
    enc.train(); proj.train(); tot = 0
    for v1, v2 in dl:
        v1, v2 = v1.to(device, non_blocking=True), v2.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            loss = nt_xent(proj(enc.encode(v1)), proj(enc.encode(v2)))
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); sched.step(); tot += loss.item()
    ssl_hist.append(tot / len(dl))
    if (ep + 1) % max(1, CFG["epochs_ssl"] // 8) == 0 or ep == CFG["epochs_ssl"] - 1:
        print(f"[SSL] ep {ep+1:>3}/{CFG['epochs_ssl']}  NT-Xent {ssl_hist[-1]:.4f}")

ssl_state = {k: v.detach().cpu().clone() for k, v in enc.state_dict().items()}
plt.figure(figsize=(5, 3)); plt.plot(ssl_hist); plt.xlabel("epoch"); plt.ylabel("NT-Xent loss")
plt.title("Contrastive pre-training"); savefig("ssl_loss"); plt.show()

In [ ]:
fracs = [0.05, 0.10, 0.25, 1.0]
perm = np.random.RandomState(SEED).permutation(len(Xtr))
rows, models = [], {}
for frac in fracs:
    idx = np.sort(perm[: int(len(Xtr) * frac)])
    Xa, Ya = Xtr[idx], Ytr[idx]
    ep = int(CFG["epochs_ft"] * max(1, 0.25 / frac))
    for init in ["scratch", "ssl"]:
        seed_everything(SEED)
        m = CNNTransformer()
        if init == "ssl":
            m.load_state_dict(ssl_state); m.head.reset_parameters()
        train_supervised(m, Xa, Ya, Xva, Yva, ep, CFG["lr_ft"], f"{init}@{frac:.0%}", aug=0.5)
        auc = macro_auc(Yte, predict_logits(m, Xte))
        rows.append(dict(label_fraction=frac, init=init, n_labels=len(idx), test_macro_auroc=auc))
        models[(init, frac)] = m
        print(f"   -> {init:8s} @ {frac:>4.0%} labels ({len(idx):>6,}): test AUROC {auc:.4f}\n")

le = pd.DataFrame(rows)
tab = le.pivot(index="label_fraction", columns="init", values="test_macro_auroc")
tab["gain (SSL - scratch)"] = tab["ssl"] - tab["scratch"]
display(tab.round(4))
results["label_efficiency"] = tab.round(4).reset_index().to_dict("records")

plt.figure(figsize=(5.5, 3.6))
for init, mk in [("scratch", "o--"), ("ssl", "s-")]:
    d_ = le[le.init == init]; plt.plot(d_.label_fraction * 100, d_.test_macro_auroc, mk, label=init)
plt.axhline(results["auc_resnet_test"], color="gray", ls=":", label="ResNet1D (100% labels)")
plt.xscale("log"); plt.xticks([5, 10, 25, 100], ["5%", "10%", "25%", "100%"])
plt.xlabel("fraction of labelled training data"); plt.ylabel("test macro-AUROC"); plt.legend()
plt.title("Self-supervised pre-training vs. label budget"); savefig("label_efficiency"); plt.show()

In [ ]:
main = models[("ssl", 1.0)]
lg_main = predict_logits(main, Xte)
lg_scratch = predict_logits(models[("scratch", 1.0)], Xte)

def bootstrap_ci(y, logits, n=300, seed=0):
    r = np.random.RandomState(seed); vals = []
    for _ in range(n):
        i = r.randint(0, len(y), len(y)); vals.append(macro_auc(y[i], logits[i]))
    return np.percentile(vals, [2.5, 97.5])

per = pd.DataFrame({"ResNet1D": macro_auc(Yte, lg_resnet, True),
                    "CNN-Transformer (scratch)": macro_auc(Yte, lg_scratch, True),
                    "CNN-Transformer (SSL)": macro_auc(Yte, lg_main, True)}, index=CLASSES)
per.loc["macro"] = per.mean()
display(per.round(4))
ci = bootstrap_ci(Yte, lg_main)
results.update(auc_main_test=float(per.loc["macro", "CNN-Transformer (SSL)"]),
               auc_main_ci95=[float(ci[0]), float(ci[1])],
               per_class_auc=per["CNN-Transformer (SSL)"].round(4).to_dict())
print(f"\nMain model test macro-AUROC {results['auc_main_test']:.4f}  (95% bootstrap CI {ci[0]:.4f}-{ci[1]:.4f})")

In [ ]:
def fit_temperature(logits, y):
    lt = torch.zeros(1, requires_grad=True); l, yt = torch.tensor(logits), torch.tensor(y)
    opt = torch.optim.LBFGS([lt], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad(); loss = F.binary_cross_entropy_with_logits(l / lt.exp(), yt); loss.backward(); return loss
    opt.step(closure); return float(lt.exp())

def ece_score(p, y, bins=15):
    p, y = p.ravel(), y.ravel(); idx = np.clip((p * bins).astype(int), 0, bins - 1); e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any(): e += m.mean() * abs(p[m].mean() - y[m].mean())
    return e

def reliability(ax, p, y, title, bins=10):
    p, y = p.ravel(), y.ravel(); idx = np.clip((p * bins).astype(int), 0, bins - 1)
    xs = [p[idx == b].mean() for b in range(bins) if (idx == b).sum() > 20]
    ys = [y[idx == b].mean() for b in range(bins) if (idx == b).sum() > 20]
    ax.plot([0, 1], [0, 1], "k:"); ax.plot(xs, ys, "o-"); ax.set_title(title)
    ax.set_xlabel("mean predicted probability"); ax.set_ylabel("observed frequency")

lg_val = predict_logits(main, Xva)
T = fit_temperature(lg_val, Yva)
p_raw, p_cal = sigmoid(lg_main), sigmoid(lg_main / T)
results.update(temperature=T, ece_before=float(ece_score(p_raw, Yte)), ece_after=float(ece_score(p_cal, Yte)))
print(f"T = {T:.3f} | ECE {results['ece_before']:.4f} -> {results['ece_after']:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
reliability(ax[0], p_raw, Yte, f"Before  (ECE {results['ece_before']:.3f})")
reliability(ax[1], p_cal, Yte, f"After temperature scaling  (ECE {results['ece_after']:.3f})")
plt.tight_layout(); savefig("reliability"); plt.show()

In [ ]:
pid = meta_te["patient_id"].values
r = np.random.RandomState(SEED); pats = r.permutation(np.unique(pid))
cal_pats = set(pats[: len(pats) // 2]); is_cal = np.array([p in cal_pats for p in pid])
p_c, y_c = p_cal[is_cal], Yte[is_cal]          # calibration half
p_e, y_e = p_cal[~is_cal], Yte[~is_cal]        # evaluation half
print(f"conformal calibration: {is_cal.sum():,} ECGs | evaluation: {(~is_cal).sum():,} ECGs (disjoint patients)")

def conformal_thresholds(p, y, alpha):
    th = []
    for c in range(K):
        s = 1 - p[y[:, c] == 1, c]; n = len(s)
        q = min(np.ceil((n + 1) * (1 - alpha)) / n, 1.0)
        th.append(1 - np.quantile(s, q, method="higher"))     # include class c iff p_c >= th[c]
    return np.array(th)

NORM_I = CLASSES.index("NORM")
def is_confident(sets):
    nonempty = sets.any(1); path = np.delete(sets, NORM_I, 1).any(1)
    return nonempty & ~(sets[:, NORM_I] & path)

rows = []
for alpha in [0.20, 0.10, 0.05]:
    th = conformal_thresholds(p_c, y_c, alpha); sets = p_e >= th
    cov = np.array([sets[y_e[:, c] == 1, c].mean() for c in range(K)])
    conf = is_confident(sets); exact = (sets == (y_e == 1)).all(1)
    rows.append(dict(alpha=alpha, target_recall=1 - alpha, min_class_coverage=cov.min(), mean_class_coverage=cov.mean(),
                     avg_set_size=sets.sum(1).mean(), pct_confident=conf.mean(),
                     exact_match_all=exact.mean(), exact_match_confident=exact[conf].mean()))
cp = pd.DataFrame(rows); display(cp.round(3))
results["conformal"] = cp.round(4).to_dict("records")

# operating point used for export & demos
ALPHA = 0.10
TH = conformal_thresholds(p_c, y_c, ALPHA)
cov_by_class = {CLASSES[c]: float((p_e >= TH)[y_e[:, c] == 1, c].mean()) for c in range(K)}
print("\nper-class coverage @ alpha=0.10 (target >= 0.90):", {k: round(v, 3) for k, v in cov_by_class.items()})

plt.figure(figsize=(4.6, 4))
plt.plot([0.75, 1], [0.75, 1], "k:", label="ideal"); plt.plot(cp.target_recall, cp.mean_class_coverage, "o-", label="empirical (mean over classes)")
plt.plot(cp.target_recall, cp.min_class_coverage, "s--", label="empirical (worst class)")
plt.xlabel("target coverage (1 - alpha)"); plt.ylabel("empirical coverage"); plt.legend(); plt.title("Conformal coverage guarantee")
savefig("conformal_coverage"); plt.show()

In [ ]:
# A single cal/eval split gives a noisy coverage estimate (esp. for small classes like HYP).
# The guarantee holds on average over calibration draws, so repeat the patient-level split and report mean +/- std.
reps = []
for s_ in range(30):
    rr = np.random.RandomState(1000 + s_); pp = rr.permutation(np.unique(pid)); cs = set(pp[: len(pp) // 2])
    m_ = np.array([p in cs for p in pid]); th_ = conformal_thresholds(p_cal[m_], Yte[m_], ALPHA); st_ = p_cal[~m_] >= th_
    reps.append([st_[Yte[~m_][:, c] == 1, c].mean() for c in range(K)])
reps = np.array(reps)
cov_tab = pd.DataFrame({"mean coverage": reps.mean(0), "std over splits": reps.std(0), "target": 1 - ALPHA}, index=CLASSES)
display(cov_tab.round(3))
results["conformal_repeated_splits"] = cov_tab.round(4).reset_index().to_dict("records")
print(f"average coverage over classes and 30 random patient-level splits: {reps.mean():.3f} (target >= {1-ALPHA:.2f})")

In [ ]:
def gradcam(model, x, cls_idx):
    model.eval().to(device); store = {}
    def hook(m, i, o):
        store["a"] = o; o.register_hook(lambda g: store.__setitem__("g", g))
    h = model.stem.register_forward_hook(hook)
    xin = x.unsqueeze(0).to(device)
    out = model(xin); model.zero_grad(); out[0, cls_idx].backward(); h.remove()
    a, g = store["a"][0].detach(), store["g"][0]
    cam = F.relu((g.mean(1, keepdim=True) * a).sum(0))
    cam = F.interpolate(cam[None, None], size=x.shape[-1], mode="linear", align_corners=False)[0, 0]
    return ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).cpu().numpy()

def plot_case(i, cls, title, leads=("II", "V2", "V5")):
    c = CLASSES.index(cls); x = torch.from_numpy(Xte[i]); cam = gradcam(main, x, c)
    fig, axs = plt.subplots(len(leads), 1, figsize=(10, 1.9 * len(leads)), sharex=True); t = np.arange(x.shape[1]) / 100
    for ax, ld in zip(np.atleast_1d(axs), leads):
        s = Xte[i, LEADS.index(ld)]; lo, hi = s.min() - .3, s.max() + .3
        ax.imshow(cam[None], aspect="auto", extent=[0, 10, lo, hi], cmap="Reds", alpha=0.45)
        ax.plot(t, s, "k", lw=0.8); ax.set_ylim(lo, hi); ax.set_ylabel(ld, rotation=0, labelpad=14); ax.grid(False)
    np.atleast_1d(axs)[-1].set_xlabel("seconds")
    truth = [CLASSES[k] for k in np.where(Yte[i] == 1)[0]]
    plt.suptitle(f"{title} | explaining '{cls}' | p={p_cal[i, c]:.2f} | true labels: {truth}", fontsize=10, y=1.0)
    plt.tight_layout(); savefig(f"gradcam_{title.replace(' ', '_')}"); plt.show()

ci_mi, ci_st = CLASSES.index("MI"), CLASSES.index("STTC")
pos_mi = np.where(Yte[:, ci_mi] == 1)[0]
plot_case(pos_mi[np.argmax(p_cal[pos_mi, ci_mi])], "MI", "Confident correct MI")
pos_st = np.where(Yte[:, ci_st] == 1)[0]
plot_case(pos_st[np.argmax(p_cal[pos_st, ci_st])], "STTC", "Confident correct STTC")
plot_case(pos_mi[np.argmin(p_cal[pos_mi, ci_mi])], "MI", "FAILURE missed MI")     # always show failures too

In [ ]:
def deletion_test(cls, n=150, frac=0.2, seed=0):
    # Mask the top-`frac` most 'relevant' time steps (all leads) vs. an equally-shaped mask at a random position.
    # If the explanation is meaningful, deleting it must hurt the prediction MORE than deleting a random region.
    c = CLASSES.index(cls); r = np.random.RandomState(seed)
    idxs = r.choice(np.where(Yte[:, c] == 1)[0], min(n, int(Yte[:, c].sum())), replace=False)
    def prob(x):
        with torch.no_grad(): return float(torch.sigmoid(main(x[None].to(device)) / T)[0, c])
    d_cam, d_rnd = [], []
    for i in tqdm(idxs, desc=f"deletion test {cls}"):
        x = torch.from_numpy(Xte[i]); cam = gradcam(main, x, c)
        m = cam >= np.quantile(cam, 1 - frac); m_r = np.roll(m, int(r.randint(100, 900)))
        base = prob(x)
        for mask, lst in [(m, d_cam), (m_r, d_rnd)]:
            xm = x.clone(); xm[:, torch.from_numpy(mask)] = 0; lst.append(base - prob(xm))
    d_cam, d_rnd = np.array(d_cam), np.array(d_rnd)
    p = wilcoxon(d_cam, d_rnd, alternative="greater").pvalue
    print(f"{cls}: mean prob. drop | Grad-CAM region {d_cam.mean():.3f} vs random region {d_rnd.mean():.3f}  (Wilcoxon p={p:.2e}, n={len(idxs)})")
    return dict(cls=cls, drop_gradcam=float(d_cam.mean()), drop_random=float(d_rnd.mean()), p_value=float(p), n=len(idxs))

results["deletion_test"] = [deletion_test(c) for c in ["MI", "STTC"]]

In [ ]:
groups = {"male": meta_te["sex"].values == 0, "female": meta_te["sex"].values == 1}
age = meta_te["age"].values
for lo, hi in [(0, 40), (40, 60), (60, 75), (75, 200)]:
    groups[f"age {lo}-{hi}" if hi < 200 else f"age {lo}+"] = (age >= lo) & (age < hi)

rows = []
for g, m in groups.items():
    a = macro_auc(Yte[m], lg_main[m], True)
    rows.append(dict(group=g, n=int(m.sum()), macro_auroc=np.nanmean(a), **{c: v for c, v in zip(CLASSES, a)}))
sub = pd.DataFrame(rows).set_index("group"); display(sub.round(3))
gap = sub.macro_auroc.max() - sub.macro_auroc.min()
print(f"largest macro-AUROC gap between subgroups: {gap:.3f}")
results["subgroups"] = sub.round(4).reset_index().to_dict("records"); results["subgroup_max_gap"] = float(gap)

plt.figure(figsize=(7, 3.2)); plt.bar(sub.index, sub.macro_auroc, color="#4c78a8")
plt.axhline(results["auc_main_test"], color="k", ls=":"); plt.ylim(max(0.5, sub.macro_auroc.min() - .05), 1)
plt.xticks(rotation=30, ha="right"); plt.ylabel("macro-AUROC"); plt.title("Subgroup performance"); savefig("subgroups"); plt.show()

In [ ]:
audit_models = {"ResNet1D": resnet, "CNN-Transformer (scratch)": models[("scratch", 1.0)], "CNN-Transformer (SSL)": main}
rs = np.random.RandomState(1)

sigmas = [0, 0.1, 0.2, 0.4, 0.8]
noise_res = {n: [macro_auc(Yte, predict_logits(m, Xte + s * rs.randn(*Xte.shape).astype(np.float32))) for s in sigmas]
             for n, m in audit_models.items()}

def drop_leads(X, k, seed):
    ranks = np.random.RandomState(seed).rand(len(X), 12).argsort(1).argsort(1)
    return X * (~(ranks < k))[:, :, None]

ks = [0, 1, 2, 4, 6]
lead_res = {n: [np.mean([macro_auc(Yte, predict_logits(m, drop_leads(Xte, k, s))) for s in range(2 if k else 1)]) for k in ks]
            for n, m in audit_models.items()}

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for n in audit_models:
    ax[0].plot(sigmas, noise_res[n], "o-", label=n); ax[1].plot(ks, lead_res[n], "o-", label=n)
ax[0].set_xlabel("added Gaussian noise (std, in z-score units)"); ax[1].set_xlabel("number of leads zeroed (random)")
for a in ax: a.set_ylabel("test macro-AUROC")
ax[0].set_title("Robustness to sensor noise"); ax[1].set_title("Robustness to missing leads"); ax[0].legend(fontsize=8)
plt.tight_layout(); savefig("robustness"); plt.show()
results["robust_noise"] = {n: dict(zip(map(str, sigmas), np.round(v, 4))) for n, v in noise_res.items()}
results["robust_leads"] = {n: dict(zip(map(str, ks), np.round(v, 4))) for n, v in lead_res.items()}

In [ ]:
ART = "/content/cardiotrust_artifacts"; os.makedirs(ART, exist_ok=True)
torch.save(main.state_dict(), f"{ART}/cardiotrust_cnn_transformer_ssl.pt")
json.dump(dict(classes=CLASSES, leads=LEADS, sampling_rate_hz=100, input_shape=[12, 1000],
               temperature=T, conformal_alpha=ALPHA, conformal_thresholds=dict(zip(CLASSES, map(float, TH))),
               norm_mean=mu.ravel().tolist(), norm_std=sd.ravel().tolist(),
               architecture=dict(name="CNNTransformer", d=128, heads=4, layers=4)),
          open(f"{ART}/config.json", "w"), indent=2)
json.dump(results, open(f"{ART}/results.json", "w"), indent=2, default=float)
!cp -r {FIG} {ART}/figures && cd /content && zip -qr cardiotrust_artifacts.zip cardiotrust_artifacts
print("saved ->", ART, "| zip: /content/cardiotrust_artifacts.zip")
# from google.colab import files; files.download("/content/cardiotrust_artifacts.zip")   # uncomment to download

r0 = le[le.label_fraction == 0.10].set_index("init").test_macro_auroc
print("\n=========== NUMBERS FOR YOUR RESUME / README (all measured on held-out test fold) ===========")
print(f"Macro-AUROC (SSL CNN-Transformer): {results['auc_main_test']:.3f}  [95% CI {results['auc_main_ci95'][0]:.3f}-{results['auc_main_ci95'][1]:.3f}]   vs ResNet1D {results['auc_resnet_test']:.3f}")
print(f"Label efficiency @10% labels: SSL {r0['ssl']:.3f} vs scratch {r0['scratch']:.3f}  (+{r0['ssl']-r0['scratch']:.3f})")
print(f"ECE: {results['ece_before']:.3f} -> {results['ece_after']:.3f} after temperature scaling (T={T:.2f})")
row = cp[cp.alpha == 0.10].iloc[0]
print(f"Conformal @ alpha=0.10: worst-class coverage {row.min_class_coverage:.3f} (target 0.90), avg set size {row.avg_set_size:.2f}, "
      f"{row.pct_confident:.0%} cases auto-handled, exact-match {row.exact_match_all:.2f} overall -> {row.exact_match_confident:.2f} on confident cases")
print(f"Largest subgroup AUROC gap: {results['subgroup_max_gap']:.3f}")
for d_ in results["deletion_test"]:
    print(f"Deletion test {d_['cls']}: Grad-CAM region drop {d_['drop_gradcam']:.3f} vs random {d_['drop_random']:.3f} (p={d_['p_value']:.1e})")